# Colab Haptic Research Starter

This notebook uploads one sample video, analyzes audio and video together, and exports `output_haptic_map.json` for manual download.

It keeps the current Android JSON contract:
- `window_size_ms`
- `track`
- `events`
- `frames`

Future ML hooks are left in place for audio and video models.

## 1) Set Up Environment and Imports

Install or verify the notebook dependencies, then import the libraries used by the starter pipeline.

In [ ]:
# Install or verify dependencies when running in Colab.
try:
    import google.colab  # type: ignore
    _running_in_colab = True
except Exception:
    _running_in_colab = False

if _running_in_colab:
    %pip install -q opencv-python-headless librosa pandas matplotlib scipy soundfile
else:
    print("Local notebook run: ensure the required packages are installed.")

In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import subprocess
import sys
import tempfile
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import google.colab  # type: ignore
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    files = None

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print(f"Colab runtime detected: {IN_COLAB}")

## 2) Define Configuration and Constants

Use one config object so paths, thresholds, and feature flags stay editable in one place.

In [ ]:
@dataclass
class ResearchConfig:
    window_size_ms: int = 40
    output_filename: str = "output_haptic_map.json"
    min_intensity: int = 16
    audio_rms_max: float = 0.5
    audio_thunder_threshold: float = 100.0
    lightning_brightness_scale_max: float = 50.0
    lightning_threshold_visual_low: float = 140.0
    lightning_threshold_visual_high: float = 160.0
    rain_diff_threshold: int = 20
    rain_ratio_max: float = 0.12
    rain_intensity_max: float = 200.0
    rain_intensity_threshold: float = 30.0
    enable_future_audio_model: bool = False
    enable_future_video_model: bool = False

    def resolve_output_path(self, video_path: Path) -> Path:
        stem = video_path.stem or "output"
        return video_path.with_name(f"{stem}_{self.output_filename}")


CONFIG = ResearchConfig()
print(CONFIG)

## 3) Create Core Data Structures

These dataclasses describe frame-level measurements, event windows, and the final JSON payload.

In [ ]:
@dataclass
class FrameMetrics:
    frame: int
    timestamp_ms: int
    brightness: float
    brightness_spike: float
    lightning_intensity: float
    rain_motion: int
    rain_ratio: float
    rain_intensity: float
    audio_rms: float
    audio_intensity: float


@dataclass
class EventWindow:
    start_ms: int
    event_type: str
    intensity: int
    max_brightness: float
    max_lightning: float
    max_rain: float
    avg_audio: float
    timestamp: str = ""


@dataclass
class HapticMapOutput:
    window_size_ms: int
    track: Dict[str, int] = field(default_factory=dict)
    events: List[Dict[str, Any]] = field(default_factory=list)
    frames: List[Dict[str, Any]] = field(default_factory=list)


def ms_to_timestamp(ms: int) -> str:
    seconds = ms // 1000
    ms_rem = ms % 1000
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    return f"{hours:02d}:{minutes:02d}:{secs:02d}.{ms_rem:03d}"

## 4) Implement Main Processing Function

The first working version keeps the current rule-based behavior and leaves hooks for future audio and video models.

In [ ]:
def validate_video_path(video_path: Path) -> None:
    if not video_path.exists():
        raise FileNotFoundError(f"Video file not found: {video_path}")
    if video_path.stat().st_size <= 0:
        raise ValueError(f"Video file is empty: {video_path}")


def resolve_ffmpeg_available() -> bool:
    try:
        subprocess.run(["ffmpeg", "-version"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return True
    except Exception:
        return False


def extract_audio_to_temp_wav(video_path: Path) -> Optional[Path]:
    if not resolve_ffmpeg_available():
        return None

    temp_audio = video_path.with_suffix(".temp_audio.wav")
    command = [
        "ffmpeg",
        "-y",
        "-i",
        str(video_path),
        "-vn",
        "-ac",
        "2",
        "-ar",
        "44100",
        str(temp_audio),
    ]
    subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return temp_audio


def classify_window(
    window_frames: Sequence[FrameMetrics],
    config: ResearchConfig,
    audio_model: Optional[Callable[[Sequence[FrameMetrics]], Tuple[str, float]]] = None,
    video_model: Optional[Callable[[Sequence[FrameMetrics]], Tuple[str, float]]] = None,
) -> EventWindow:
    max_lightning = max(frame.lightning_intensity for frame in window_frames)
    max_brightness = max(frame.brightness for frame in window_frames)
    max_rain = max(frame.rain_intensity for frame in window_frames)
    avg_audio = float(np.mean([frame.audio_intensity for frame in window_frames]))

    model_audio_label, model_audio_score = ("", 0.0)
    model_video_label, model_video_score = ("", 0.0)
    if audio_model is not None and config.enable_future_audio_model:
        model_audio_label, model_audio_score = audio_model(window_frames)
    if video_model is not None and config.enable_future_video_model:
        model_video_label, model_video_score = video_model(window_frames)

    event_type = "none"
    intensity = 0

    if model_audio_label or model_video_label:
        if model_audio_label:
            event_type = model_audio_label
            intensity = int(np.clip(model_audio_score * 255, 0, 255))
        if model_video_label and model_video_score >= model_audio_score:
            event_type = model_video_label
            intensity = int(np.clip(model_video_score * 255, 0, 255))
    else:
        if max_lightning > config.lightning_threshold_visual_low and avg_audio > config.audio_thunder_threshold:
            event_type = "thunder_both"
            intensity = 255
        elif max_lightning > config.lightning_threshold_visual_high:
            event_type = "thunder_visual"
            intensity = 255
        elif max_rain > config.rain_intensity_threshold:
            event_type = "rain"
            intensity = int(max_rain)
        elif avg_audio > config.min_intensity:
            event_type = "audio"
            intensity = int(avg_audio)

    if intensity < config.min_intensity:
        intensity = 0

    return EventWindow(
        start_ms=window_frames[0].timestamp_ms,
        event_type=event_type,
        intensity=int(intensity),
        max_brightness=float(max_brightness),
        max_lightning=float(max_lightning),
        max_rain=float(max_rain),
        avg_audio=float(avg_audio),
        timestamp=ms_to_timestamp(window_frames[0].timestamp_ms),
    )


def collect_frame_metrics(video_path: Path, config: ResearchConfig) -> Tuple[List[FrameMetrics], float, int, int, int]:
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    fps = capture.get(cv2.CAP_PROP_FPS) or 0.0
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    frame_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    frame_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    if fps <= 0 or total_frames <= 0:
        raise ValueError(f"Invalid video metadata: fps={fps}, total_frames={total_frames}")

    audio_rms = np.zeros(total_frames, dtype=np.float32)
    temp_audio = None
    try:
        temp_audio = extract_audio_to_temp_wav(video_path)
        if temp_audio is not None and temp_audio.exists():
            audio_waveform, sample_rate = librosa.load(str(temp_audio), sr=None)
            frame_length = max(int(sample_rate / fps), 1)
            audio_rms = librosa.feature.rms(y=audio_waveform, frame_length=frame_length, hop_length=frame_length)[0]
    finally:
        if temp_audio is not None and temp_audio.exists():
            temp_audio.unlink(missing_ok=True)

    frames: List[FrameMetrics] = []
    previous_downsampled: Optional[np.ndarray] = None
    previous_brightness = 0.0

    for frame_index in range(total_frames):
        ok, frame = capture.read()
        if not ok:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        downsampled = cv2.resize(gray, (max(frame_width // 4, 1), max(frame_height // 4, 1)))
        if previous_downsampled is None:
            previous_downsampled = downsampled.copy()

        diff = cv2.absdiff(downsampled, previous_downsampled)
        _, diff_threshold = cv2.threshold(diff, config.rain_diff_threshold, 255, cv2.THRESH_BINARY)
        rain_motion = int(np.count_nonzero(diff_threshold))
        previous_downsampled = downsampled.copy()

        avg_brightness = float(np.mean(gray))
        brightness_spike = max(0.0, avg_brightness - previous_brightness)
        previous_brightness = avg_brightness

        audio_value = float(audio_rms[frame_index]) if frame_index < len(audio_rms) else 0.0
        audio_intensity = float(np.interp(audio_value, [0, config.audio_rms_max], [0, 255]))
        lightning_intensity = float(np.interp(brightness_spike, [0.0, config.lightning_brightness_scale_max], [0, 255]))
        rain_ratio = float(rain_motion / (downsampled.shape[0] * downsampled.shape[1]))
        rain_intensity = float(np.interp(rain_ratio, [0.0, config.rain_ratio_max], [0, config.rain_intensity_max]))
        timestamp_ms = int((frame_index / fps) * 1000)

        frames.append(
            FrameMetrics(
                frame=frame_index,
                timestamp_ms=timestamp_ms,
                brightness=avg_brightness,
                brightness_spike=brightness_spike,
                lightning_intensity=lightning_intensity,
                rain_motion=rain_motion,
                rain_ratio=rain_ratio,
                rain_intensity=rain_intensity,
                audio_rms=audio_value,
                audio_intensity=audio_intensity,
            )
        )

    capture.release()
    return frames, fps, total_frames, frame_width, frame_height


def build_haptic_map_from_frames(
    frames: Sequence[FrameMetrics],
    config: ResearchConfig,
    audio_model: Optional[Callable[[Sequence[FrameMetrics]], Tuple[str, float]]] = None,
    video_model: Optional[Callable[[Sequence[FrameMetrics]], Tuple[str, float]]] = None,
) -> HapticMapOutput:
    if not frames:
        raise ValueError("No frame metrics supplied")

    windows: List[EventWindow] = []
    bucket: List[FrameMetrics] = []
    current_window_start = (frames[0].timestamp_ms // config.window_size_ms) * config.window_size_ms

    for frame in frames:
        frame_window_start = (frame.timestamp_ms // config.window_size_ms) * config.window_size_ms
        if frame_window_start != current_window_start and bucket:
            windows.append(classify_window(bucket, config, audio_model=audio_model, video_model=video_model))
            bucket = []
            current_window_start = frame_window_start
        bucket.append(frame)

    if bucket:
        windows.append(classify_window(bucket, config, audio_model=audio_model, video_model=video_model))

    haptic_map = HapticMapOutput(window_size_ms=config.window_size_ms)
    for window in windows:
        haptic_map.track[str(window.start_ms)] = window.intensity
        haptic_map.events.append(
            {
                "start_ms": window.start_ms,
                "type": window.event_type,
                "intensity": window.intensity,
                "max_brightness": window.max_brightness,
                "max_lightning": window.max_lightning,
                "max_rain": window.max_rain,
                "avg_audio": window.avg_audio,
                "timestamp": window.timestamp,
            }
        )
    haptic_map.frames = [asdict(frame) for frame in frames]
    return haptic_map


def process_video_to_haptic_map(
    video_path: Path,
    config: ResearchConfig = CONFIG,
    audio_model: Optional[Callable[[Sequence[FrameMetrics]], Tuple[str, float]]] = None,
    video_model: Optional[Callable[[Sequence[FrameMetrics]], Tuple[str, float]]] = None,
) -> HapticMapOutput:
    validate_video_path(video_path)
    frames, fps, total_frames, frame_width, frame_height = collect_frame_metrics(video_path, config)
    print(f"Loaded video: fps={fps:.2f}, frames={total_frames}, size={frame_width}x{frame_height}")
    map_output = build_haptic_map_from_frames(frames, config, audio_model=audio_model, video_model=video_model)

    output_path = config.resolve_output_path(video_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as handle:
        json.dump(asdict(map_output), handle, indent=2)

    print(f"Saved JSON to {output_path}")
    return map_output

## 5) Add Input Validation and Error Handling

Raise clear errors for missing clips, invalid metadata, and empty frame lists.

In [ ]:
def summarize_haptic_map(haptic_map: HapticMapOutput, limit: int = 10) -> pd.DataFrame:
    events = pd.DataFrame(haptic_map.events)
    if events.empty:
        return events
    columns = [col for col in ["start_ms", "timestamp", "type", "intensity", "max_lightning", "max_rain", "avg_audio"] if col in events.columns]
    return events.loc[:, columns].head(limit)


def ensure_map_has_content(haptic_map: HapticMapOutput) -> None:
    if not haptic_map.events:
        raise ValueError("Haptic map contains no events")
    if not haptic_map.frames:
        raise ValueError("Haptic map contains no frame metrics")
    if not any(value > 0 for value in haptic_map.track.values()):
        print("Warning: track contains only zeros")


print("Validation helpers loaded.")

## 6) Run a Minimal End-to-End Example

Use a synthetic short clip of frame metrics first, then run the real upload flow in the next section.

In [ ]:
sample_frames = [
    FrameMetrics(0, 0, 12.0, 0.0, 0.0, 0, 0.0, 0.0, 0.01, 5.0),
    FrameMetrics(1, 40, 18.0, 6.0, 60.0, 15, 0.02, 18.0, 0.08, 20.0),
    FrameMetrics(2, 80, 24.0, 6.0, 120.0, 42, 0.05, 48.0, 0.15, 58.0),
    FrameMetrics(3, 120, 35.0, 11.0, 190.0, 60, 0.08, 72.0, 0.18, 90.0),
    FrameMetrics(4, 160, 20.0, 0.0, 22.0, 3, 0.01, 8.0, 0.03, 10.0),
]

sample_map = build_haptic_map_from_frames(sample_frames, CONFIG)
ensure_map_has_content(sample_map)
print(json.dumps(asdict(sample_map), indent=2)[:1200])
display(summarize_haptic_map(sample_map))

## 7) Add Basic Unit Tests in Notebook Cells

These simple asserts verify timestamp formatting, validation, and JSON generation helpers.

In [ ]:
assert ms_to_timestamp(0) == "00:00:00.000"
assert ms_to_timestamp(1234) == "00:00:01.234"
assert CONFIG.window_size_ms == 40
assert CONFIG.min_intensity == 16

try:
    validate_video_path(Path("/content/does_not_exist.mp4"))
    raise AssertionError("Expected FileNotFoundError")
except FileNotFoundError:
    pass

assert summarize_haptic_map(sample_map).shape[0] >= 1
assert any(event["intensity"] >= 0 for event in sample_map.events)
print("Basic notebook tests passed.")

## 8) Upload a Real Video and Export JSON

Upload your 1.30-minute sample clip, process it, inspect the event table, and download the JSON manually.

In [ ]:
def save_uploaded_bytes(uploaded: Dict[str, bytes]) -> Path:
    if not uploaded:
        raise ValueError("No file uploaded")
    filename, content = next(iter(uploaded.items()))
    target = Path("/content") / filename
    target.write_bytes(content)
    return target


if IN_COLAB:
    uploaded_files = files.upload()
    sample_video_path = save_uploaded_bytes(uploaded_files)
    result_map = process_video_to_haptic_map(sample_video_path, CONFIG)
    print("Top events preview:")
    display(summarize_haptic_map(result_map, limit=15))
    output_json_path = CONFIG.resolve_output_path(sample_video_path)
    print(f"JSON ready for manual download: {output_json_path}")
    # Uncomment the next line when you want Colab to trigger the download dialog.
    # files.download(str(output_json_path))
else:
    print("This upload flow is meant for Google Colab.")